## RAG PIPELINE
#### DATA INGESTION TO VECTOR DB

In [153]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [154]:
##DATA INGRSTION
##READ ALL THE PDFS INSIDE THE DIRECTORY
def Read_all_pdfs(pdf_directory):
    """"Processes all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found{len(pdf_files)}PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            #ADD source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"---Loaded---{len(documents)}pages")
        except Exception as e:
            print(f"ERROR:{e}")

    print(f"\nTotal documents loaded:{len(all_documents)}")
    return all_documents

#process all PDFs in the DATA Directory
all_pdf_documents=Read_all_pdfs("../data")


Found3PDF files to process

Processing:ai_basics.pdf
---Loaded---1pages

Processing:climate_report.pdf
---Loaded---1pages

Processing:python_guide.pdf
---Loaded---1pages

Total documents loaded:3


In [155]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-13T18:21:42+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-13T18:21:42+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\pdf\\ai_basics.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_basics.pdf', 'file_type': 'pdf'}, page_content='Artificial Intelligence Basics\nWhat is AI?\nArtificial Intelligence is the simulation of human intelligence by machines. It enables computers to learn\nfrom data, recognize patterns, and make decisions with minimal human intervention.\nMachine Learning\nMachine Learning is a subset of AI where systems learn from data automatically. Instead of being\nexplicitly programmed, ML models improve through experience. Common types include supervised\nlearning, unsupervised learning, and reinforcement learning.\nDeep Learning\nDeep Lea

In [194]:
### chunking
### splitting
def split_doc(documents,chunk_size=500,chunk_overlap=100):
    """"split documents into smaller chunks for better RAG preferences"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into{len(split_docs)}chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}")
    return split_docs

In [195]:
chunk=split_doc(all_pdf_documents)

Split3 documents into9chunks

Example chunk:
Content:Artificial Intelligence Basics
What is AI?
Artificial Intelligence is the simulation of human intelligence by machines. It enables computers to learn
from data, recognize patterns, and make decisions ...
Metadata:{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-13T18:21:42+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-13T18:21:42+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\pdf\\ai_basics.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_basics.pdf', 'file_type': 'pdf'}


### EMBEDDING

In [196]:
### Embediing
### 2ND LAST STEP
import numpy as np
from sentence_transformers import SentenceTransformer ##embediing
import chromadb
from chromadb.config import Settings
import uuid ## vector db have ids
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity




In [197]:
class EmbeddingManager:
    """"Handles Document Embedding Generation using SentenceTransformer"""
    def __init__(self,model_name:str ="all-MiniLM-L6-v2"):
        """
        Initiliazing the embedding manager
        Args:
             model name :HuggingFace model name for sentence embedding
        """
        self.model_name=model_name
        self.model=None
        self._load_model()     


    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfuly.Embedding dimensions:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:  
            print(f"Error loading model{self.model_name}:{e}")
            raise 

    def generate_embeddings(self,texts :List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts:
        Args:
            texts:list of text strings to embed

        Returns:
            numpy array of embeddings with shape(len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not Loaded")
        print(f"Generating embeddings for {len(texts)}texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings    


    def get_embedding_dimensions(self)->int:
        """Get the embedding dimension of the model """
        if not self.model:
            raise ValueError("Model not loaded") 
        return self>model>get_sentence_embedding_dimension()           

### initlize embedding manager
embeding_manager=EmbeddingManager()

Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1629.19it/s]


Model loaded successfuly.Embedding dimensions:384


C:\Users\Abdullah\AppData\Local\Temp\ipykernel_16964\2092121526.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfuly.Embedding dimensions:{self.model.get_sentence_embedding_dimension()}")


In [198]:
embeding_manager

### VECTOR STORE

In [199]:
### vector store
class VectorStore:
    """ Manages document embedding in a chromeDB vector store"""
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name:Name of the ChromeDB collection
            persist_directory:Directory to persist the vector DB
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()

    
    def _initialize_store(self):
        """Initialize chromeDB client and collection"""
        try:
            # Create persistent ChromeDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
             
            #Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized.Collection:{self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")

        except Exception as e:
            print(f"Error initilizing vector store:{0}")
            raise

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
        documents:List of LangChain documents
        embeddings:Corresponding embeddings for the documents"""
        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)}documenst to vector store....")

        # Prepare data for ChromeDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        collection=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique ID
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # preparing metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)  
# Document content
            documents_text.append(doc.page_content)
            # Embedding
            embeddings_list.append(embedding.tolist())
            #add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added{len(documents)}documents to vector store")
            print(f"Total documents in collections:{self.collection.count()}")

        except Exception as e:
            print(f"Error addline documents to vector store:{e}")
            raise
vectorstore=VectorStore()

Vector store initialized.Collection:pdf_documents
Existing documents in collection:138


In [200]:
chunk

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-13T18:21:42+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-13T18:21:42+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\pdf\\ai_basics.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'ai_basics.pdf', 'file_type': 'pdf'}, page_content='Artificial Intelligence Basics\nWhat is AI?\nArtificial Intelligence is the simulation of human intelligence by machines. It enables computers to learn\nfrom data, recognize patterns, and make decisions with minimal human intervention.\nMachine Learning\nMachine Learning is a subset of AI where systems learn from data automatically. Instead of being\nexplicitly programmed, ML models improve through experience. Common types include supervised\nlearning, unsupervised learning, and reinforcement learning.'),
 Document(metadata={'

In [201]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunk]
texts
### Generate the embeddings 
embeddings=embeding_manager.generate_embeddings(texts)
## store in vector db
vectorstore.add_documents(chunk,embeddings)

Generating embeddings for 9texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.87it/s]


Adding 9documenst to vector store....
Successfully added9documents to vector store
Total documents in collections:147


In [202]:
### RETRIVAL

In [203]:
def retrieve_documents(query: str, n_results: int = 3):
    # 1. embed the query
    query_embedding = embeding_manager.generate_embeddings([query])
    
    # 2. search chromadb
    results = vectorstore.collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    
    return results

# test it
results = retrieve_documents("What is machine learning?")
print(results)

Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.03it/s]

{'ids': [['doc_6f924a90_1', 'doc_a37202a1_1', 'doc_aca3648e_1']], 'embeddings': None, 'documents': [['from data, recognize patterns, and make decisions with minimal human intervention.\nMachine Learning\nMachine Learning is a subset of AI where systems learn from data automatically. Instead of being', 'from data, recognize patterns, and make decisions with minimal human intervention.\nMachine Learning\nMachine Learning is a subset of AI where systems learn from data automatically. Instead of being', 'from data, recognize patterns, and make decisions with minimal human intervention.\nMachine Learning\nMachine Learning is a subset of AI where systems learn from data automatically. Instead of being']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'producer': 'ReportLab PDF Library - (opensource)', 'creationdate': '2026-07-13T18:21:42+00:00', 'file_type': 'pdf', 'author': '(anonymous)', 'source': '..\\data\\text_files\\pdf\\ai_basics.pdf'

### RAG RETRIVAL PIPELINE

In [204]:
class Ragretriver:
    """"handles query-based retrieval from the vector store"""

    def __init__(self,vectorstore:VectorStore,embeding_manager:EmbeddingManager):
        """
        initialize the retriver
        ARGS:
            vector_store:Vector store containing document embeddings
            embedding_manager:Manager for gathering querey embeddings
        """
        self.vectorstore=vectorstore
        self.embeding_manager=embeding_manager   

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevant documents from a querey
        Args:
            querey:The search querey
            top_k:Number of top results to return
            score_threshold:Minimum similarity score threshold

        Returns:
            List of dictinaries containing retrieved documents and metadata
            """
        print(f"Retrieving documents for querey:'{query}'")
        print(f"Top K:{top_k},Score threshold:{score_threshold}")


        # Generate querey embedding
        query_embedding=self.embeding_manager.generate_embeddings([query])[0]

        #Search in Vector Store
        try:
            results=self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k)

            #processed docs
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):

                    #convert distance into similarity score
                    similarity_score=1-distance
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i + 1
                        
                    
                        })
                    
                    else:

                        print("no documents found")
                        print(f"Retrieved{len(retrieved_docs)}documents(after filtering)")
                    return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval:{e}")
        return[]        

rag_retrieve=Ragretriver(vectorstore,embeding_manager) 

In [205]:
rag_retrieve

In [206]:
rag_retrieve.retrieve("what is python useful")


Retrieving documents for querey:'what is python useful'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.87it/s]


[{'id': 'doc_0877ce5d_6',
  'content': 'Python Programming Guide\nIntroduction to Python\nPython is a high-level, interpreted programming language known for its simplicity and readability. It was\ncreated by Guido van Rossum and first released in 1991. Python supports multiple programming\nparadigms including procedural, object-oriented, and functional programming.\nData Types\nPython has several built-in data types including integers, floats, strings, lists, tuples, dictionaries, and',
  'metadata': {'creator': '(unspecified)',
   'creationdate': '2026-07-13T18:21:42+00:00',
   'total_pages': 1,
   'producer': 'ReportLab PDF Library - (opensource)',
   'source': '..\\data\\text_files\\pdf\\python_guide.pdf',
   'trapped': '/False',
   'keywords': '',
   'subject': '(unspecified)',
   'content_length': 443,
   'author': '(anonymous)',
   'title': '(anonymous)',
   'doc_index': 6,
   'page_label': '1',
   'page': 0,
   'moddate': '2026-07-13T18:21:42+00:00',
   'file_type': 'pdf',
   's

In [207]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

prompt = ChatPromptTemplate.from_template("""
Answer ONLY based on the context below.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
""")

def generate_answer(query: str, retrieved_docs: list) -> str:
    context = "\n\n".join([doc['content'] for doc in retrieved_docs])
    chain = prompt | llm
    response = chain.invoke({
        "context": context,
        "question": query
    })
    return response.content

# test it
query = "what is python useful for?"
retrieved_docs = rag_retrieve.retrieve(query)
answer = generate_answer(query, retrieved_docs)
print(answer)

Retrieving documents for querey:'what is python useful for?'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.08it/s]


I don't know.


In [209]:
query = "what is python useful for?"
retrieved_docs =rag_retrieve.retrieve(query)
print(retrieved_docs)

Retrieving documents for querey:'what is python useful for?'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.49it/s]

[{'id': 'doc_0877ce5d_6', 'content': 'Python Programming Guide\nIntroduction to Python\nPython is a high-level, interpreted programming language known for its simplicity and readability. It was\ncreated by Guido van Rossum and first released in 1991. Python supports multiple programming\nparadigms including procedural, object-oriented, and functional programming.\nData Types\nPython has several built-in data types including integers, floats, strings, lists, tuples, dictionaries, and', 'metadata': {'moddate': '2026-07-13T18:21:42+00:00', 'author': '(anonymous)', 'source_file': 'python_guide.pdf', 'source': '..\\data\\text_files\\pdf\\python_guide.pdf', 'title': '(anonymous)', 'subject': '(unspecified)', 'content_length': 443, 'page': 0, 'creationdate': '2026-07-13T18:21:42+00:00', 'producer': 'ReportLab PDF Library - (opensource)', 'total_pages': 1, 'keywords': '', 'file_type': 'pdf', 'trapped': '/False', 'page_label': '1', 'doc_index': 6, 'creator': '(unspecified)'}, 'similarity_score'

In [ ]:
query = "what is python useful for?"
retrieved_docs = rag_retrieve.retrieve(query)
answer = generate_answer(query, retrieved_docs)
print(answer)

Retrieving documents for querey:'what is python useful for?'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.74it/s]


I don't know.


In [221]:
def generate_answer(query: str, retrieved_docs: list) -> str:
    context = "\n\n".join([doc['content'] for doc in retrieved_docs])
    
    
    chain = prompt | llm
    response = chain.invoke({
        "context": context,
        "question": query
    })
    return response.content

In [222]:
query = "who created python?"

In [223]:
query = "what is python useful for?"
retrieved_docs = rag_retrieve.retrieve(query)
answer = generate_answer(query, retrieved_docs)
print(answer)

Retrieving documents for querey:'what is python useful for?'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.45it/s]


I don't know.


In [224]:
def run_chatbot():
    print("PDF Chatbot Ready. Type 'quit' to exit.\n")
    while True:
        query = input("You: ")
        if query.lower() == "quit":
            break
        retrieved_docs = rag_retrieve.retrieve(query)
        answer = generate_answer(query, retrieved_docs)
        print(f"Bot: {answer}\n")

run_chatbot()


PDF Chatbot Ready. Type 'quit' to exit.

Retrieving documents for querey:'what is python'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.42it/s]


Bot: Python is a high-level, interpreted programming language known for its simplicity and readability.

Retrieving documents for querey:'what is machine learning'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.81it/s]


Bot: Machine Learning is a subset of AI where systems learn from data automatically.

Retrieving documents for querey:'what is 2+2'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


...

no documents found
Retrieved0documents(after filtering)


Bot: I don't know.

Retrieving documents for querey:'quir'
Top K:5,Score threshold:0.0
Generating embeddings for 1texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.29it/s]

no documents found
Retrieved0documents(after filtering)


Bot: I don't know.

